In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import psycopg2

# Load environment variables
load_dotenv(dotenv_path="../.env")

# Database connection function
def get_connection():
    return psycopg2.connect(
        host=os.getenv("DB_HOST", "localhost"),
        port=os.getenv("DB_PORT", "5432"),
        database=os.getenv("DB_NAME", "medical_dwh"),
        user=os.getenv("DB_USER", "postgres"),
        password=os.getenv("DB_PASSWORD")
    )

print("Environment loaded. Ready to query PostgreSQL!")

In [ ]:
conn = get_connection()

query_visuals = """
SELECT 
    c.channel_friendly_name,
    COUNT(f.message_id) as total_posts,
    SUM(CASE WHEN f.has_media = TRUE THEN 1 ELSE 0 END) as posts_with_images,
    ROUND(AVG(f.views_count), 2) as avg_views
FROM analytics.fct_messages f
JOIN analytics.dim_channels c ON f.channel_key = c.channel_key
GROUP BY c.channel_friendly_name;
"""

df_visuals = pd.read_sql(query_visuals, conn)
conn.close()

# Display Table
display(df_visuals)

# Plotting the visual content ratio
df_visuals['visual_ratio'] = df_visuals['posts_with_images'] / df_visuals['total_posts']
plt.figure(figsize=(10, 5))
sns.barplot(data=df_visuals, x='channel_friendly_name', y='visual_ratio', palette='viridis')
plt.title('Proportion of Visual Content (Images) by Channel')
plt.ylabel('Ratio of Posts with Images')
plt.xlabel('Channel Name')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
conn = get_connection()

# Since YOLO classes like 'person' indicate promotional/lifestyle vs. just 'bottle' (product display)
query_yolo_engagement = """
SELECT 
    CASE 
        WHEN LOWER(detected_class) = 'person' THEN 'Promotional / Lifestyle'
        WHEN LOWER(detected_class) IN ('bottle', 'cup', 'bowl') THEN 'Product Display'
        ELSE 'Other'
    END as post_category,
    COUNT(DISTINCT image_path) as unique_images,
    ROUND(AVG(views_count), 2) as average_views,
    ROUND(AVG(forwards_count), 2) as average_forwards
FROM raw.image_detections d
JOIN analytics.fct_messages f ON d.channel_name = f.channel_name AND f.image_path = d.image_path
GROUP BY 1;
"""

df_engagement = pd.read_sql(query_yolo_engagement, conn)
conn.close()

# Display Table
display(df_engagement)

# Visualizing the comparison of engagement (views)
plt.figure(figsize=(10, 5))
sns.barplot(data=df_engagement, x='post_category', y='average_views', palette='coolwarm')
plt.title('Average Views: Promotional/Lifestyle vs. Product Display Posts')
plt.ylabel('Average View Count')
plt.xlabel('Visual Post Category (YOLO Classifications)')
plt.tight_layout()
plt.show()

In [ ]:
conn = get_connection()

query_trends = """
SELECT 
    d.day_name,
    d.day_of_week,
    COUNT(f.message_id) as post_count,
    SUM(f.views_count) as total_views
FROM analytics.fct_messages f
JOIN analytics.dim_dates d ON f.date_key = d.date_key
GROUP BY d.day_name, d.day_of_week
ORDER BY d.day_of_week;
"""

df_trends = pd.read_sql(query_trends, conn)
conn.close()

# Display Table
display(df_trends)

# Plotting posting patterns across the week
plt.figure(figsize=(10, 5))
sns.lineplot(data=df_trends, x='day_name', y='post_count', marker='o', color='crimson', sort=False)
plt.title('Weekly Post Frequency Trends')
plt.ylabel('Number of Posts')
plt.xlabel('Day of the Week')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()